# 1. Configuração do Ambiente e Ingestão de Dados (Mapeamento Automático)

Nesta etapa, preparamos as dependências (removendo pacotes não utilizados para aliviar a memória) e montamos o Google Drive.
A ingestão dos arquivos foi reestruturada: ao invés de sobrescrever variáveis, o script extrai o "nome base" de cada arquivo (removendo as extensões `.las` e `.csv`). Isso permite que os arquivos originais LAS, os de eletrofácies e as tabelas de litologia sejam vinculados corretamente através de dicionários, evitando a mistura de dados entre poços distintos.


In [7]:
# Instalação apenas do necessário
!pip install lasio -q

import pandas as pd
import lasio
import os
import numpy as np
from google.colab import drive

# Monta o Google Drive
drive.mount('/content/drive')

# --- CONFIGURAÇÕES DE DIRETÓRIOS ---
PASTA_LAS_COMPLETO = "/content/drive/MyDrive/Completo"
PASTA_LAS_ELETRO = "/content/drive/MyDrive/Eletrofácies"
PASTA_CSV = r"/content/drive/MyDrive/Litologias exportadas ANA7/congresso"

dicionario_pocos = {}
dicionario_csv = {}

print("--- CARREGANDO ARQUIVOS LAS ---")
for arquivo in os.listdir(PASTA_LAS_COMPLETO):
    if arquivo.lower().endswith(".las"):
        nome_base = os.path.splitext(arquivo)[0] # Ex: "Poco_01" (usado como chave de pareamento)
        caminho_completo = os.path.join(PASTA_LAS_COMPLETO, arquivo)
        caminho_eletro = os.path.join(PASTA_LAS_ELETRO, arquivo)

        try:
            completo = lasio.read(caminho_completo)
            # Verifica e importa a curva de eletrofácies se o arquivo correspondente existir
            if os.path.exists(caminho_eletro):
                eletrofacies = lasio.read(caminho_eletro)
                if 'CLASSIFICATION_IPSOM' in eletrofacies.curves:
                    completo.append_curve('Eletrofacies', eletrofacies['CLASSIFICATION_IPSOM'].data, descr='Eletrofacies')
                    print(f"  [OK] Eletrofácies integrada: {arquivo}")

            dicionario_pocos[nome_base] = completo
        except Exception as e:
            print(f"  [Erro] Falha ao processar LAS {arquivo}: {e}")

print("\n--- CARREGANDO ARQUIVOS CSV ---")
for arquivo in os.listdir(PASTA_CSV):
    if arquivo.lower().endswith(".csv"):
        nome_base = os.path.splitext(arquivo)[0] # A chave deve bater com a do LAS
        caminho_csv = os.path.join(PASTA_CSV, arquivo)

        try:
            df = pd.read_csv(caminho_csv, encoding='latin-1', sep=';')
            dicionario_csv[nome_base] = df
            print(f"  [OK] CSV carregado: {arquivo}")
        except Exception as e:
            print(f"  [Erro] Falha ao ler CSV {arquivo}: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- CARREGANDO ARQUIVOS LAS ---
  [OK] Eletrofácies integrada: 1-SPS-96-SP_BASE.las
  [OK] Eletrofácies integrada: 1-SPS-55-SP_BASE.las
  [OK] Eletrofácies integrada: 7-SPH-14D-SPS_BASE.las
  [OK] Eletrofácies integrada: 7-SPH-15D-SPS_BASE.las
  [OK] Eletrofácies integrada: 7-SPH-16D-SPS_BASE.las
  [OK] Eletrofácies integrada: 3-SPS-69-SP_BASE.las
  [OK] Eletrofácies integrada: 7-SPH-1-SPS_BASE.las
  [OK] Eletrofácies integrada: 7-SPH-17-SPS_BASE.las
  [OK] Eletrofácies integrada: 3-SPS-82A-SP_BASE.las
  [OK] Eletrofácies integrada: 7-SPH-22-SPS_BASE.las
  [OK] Eletrofácies integrada: 7-SPH-2D-SPS_BASE.las
  [OK] Eletrofácies integrada: 7-SPH-3-SPS_BASE.las
  [OK] Eletrofácies integrada: 7-SPH-4D-SPS_BASE.las
  [OK] Eletrofácies integrada: 7-SPH-20D-SPS_BASE.las
  [OK] Eletrofácies integrada: 7-SPH-5-SPS_BASE.las
  [OK] Eletrofácies integrada: 8-SPH-19D-SPS_B

# 2. Interpolação de Litologia e Exportação Segura

Esta etapa realiza o cruzamento de dados litológicos.
*   **Segurança Matemática:** A função `preencher_array_robusto` abandonou o uso de `searchsorted` e agora aplica **máscaras booleanas**. Isso garante que as associações de fácies sejam cravadas nos intervalos exatos de topo e base, mesmo que os dados de profundidade do LAS não estejam perfeitamente ordenados.
*   **Limpeza de Strings:** O script aplica `.strip()` nas strings lidas do CSV, garantindo que espaços invisíveis não quebrem o mapeamento com o dicionário de litologias.
*   **Escrita Segura:** O script agora verifica automaticamente se o diretório de destino existe e o cria, caso necessário, evitando o travamento (FileNotFoundError) no momento de salvar o resultado.

In [8]:
# Mapa de conversão de litologia
map_facies = {"Ret": 1, "Est": 2, "Trs": 3, "Lam": 4}

def preencher_array_robusto(las, df, dicionario_map):
    """Mapeia os intervalos utilizando máscaras booleanas, imune à ordenação do LAS."""
    nome_curva_profundidade = "DEPTH"

    if nome_curva_profundidade not in las.curves.keys():
        raise KeyError(f"A curva '{nome_curva_profundidade}' não existe no LAS.")

    profundidade = np.array(las[nome_curva_profundidade])
    # Inicializa com a flag de valor nulo
    AF = np.full(len(profundidade), -999)

    for _, row in df.iterrows():
        topo = row['Topo Perfil (m)']
        base = row['Base Perfil']

        # O .strip() garante que espaços invisíveis (ex: 'Ret ') não quebrem o mapeamento
        tipo_face = str(row['Associações de Facies']).strip()
        valor_face = dicionario_map.get(tipo_face, -1)

        # A máscara avalia todos os pontos simultaneamente. É mais robusta que o searchsorted.
        mask = (profundidade >= topo) & (profundidade <= base)
        AF[mask] = valor_face

    return AF

# --- EXECUÇÃO E EXPORTAÇÃO ---
PASTA_SAIDA = "/content/drive/MyDrive/Arquivos_Modificados"
os.makedirs(PASTA_SAIDA, exist_ok=True) # Trava de segurança: Cria a pasta se não existir

print("\n--- INICIANDO CRUZAMENTO E EXPORTAÇÃO ---")
for nome_base, las_obj in dicionario_pocos.items():
    # Verifica se há um CSV correspondente para aquele poço específico
    if nome_base in dicionario_csv:
        print(f"Processando integração litológica para: {nome_base}")
        df_correspondente = dicionario_csv[nome_base]

        try:
            # Gera o vetor numérico com base nos intervalos
            AF_array = preencher_array_robusto(las_obj, df_correspondente, map_facies)

            # Se a coluna AF já existir por processamentos anteriores, remove para evitar duplicatas
            if 'AF' in las_obj.curves.keys():
                las_obj.delete_curve('AF')

            # Insere a curva formatada no objeto LAS
            las_obj.append_curve('AF', AF_array, unit="", descr="ASSOCIAÇÃO DE FACIES")

            # Salva o arquivo (Adiciona o sufixo Q, conforme o padrão original)
            nome_arquivo_saida = f"{nome_base}Q.las"
            caminho_saida = os.path.join(PASTA_SAIDA, nome_arquivo_saida)
            las_obj.write(caminho_saida, version=2.0)
            print(f"  [Sucesso] Salvo em: {caminho_saida}")

        except Exception as e:
            print(f"  [Erro] Falha ao processar e salvar {nome_base}: {e}")
    else:
        print(f"  [Pulo] Nenhum CSV correspondente encontrado para o poço {nome_base}.")


--- INICIANDO CRUZAMENTO E EXPORTAÇÃO ---
  [Pulo] Nenhum CSV correspondente encontrado para o poço 1-SPS-96-SP_BASE.
  [Pulo] Nenhum CSV correspondente encontrado para o poço 1-SPS-55-SP_BASE.
  [Pulo] Nenhum CSV correspondente encontrado para o poço 7-SPH-14D-SPS_BASE.
Processando integração litológica para: 7-SPH-15D-SPS_BASE
  [Sucesso] Salvo em: /content/drive/MyDrive/Arquivos_Modificados/7-SPH-15D-SPS_BASEQ.las
  [Pulo] Nenhum CSV correspondente encontrado para o poço 7-SPH-16D-SPS_BASE.
  [Pulo] Nenhum CSV correspondente encontrado para o poço 3-SPS-69-SP_BASE.
  [Pulo] Nenhum CSV correspondente encontrado para o poço 7-SPH-1-SPS_BASE.
  [Pulo] Nenhum CSV correspondente encontrado para o poço 7-SPH-17-SPS_BASE.
  [Pulo] Nenhum CSV correspondente encontrado para o poço 3-SPS-82A-SP_BASE.
  [Pulo] Nenhum CSV correspondente encontrado para o poço 7-SPH-22-SPS_BASE.
  [Pulo] Nenhum CSV correspondente encontrado para o poço 7-SPH-2D-SPS_BASE.
Processando integração litológica para: 